# Welcome to Subconscious!

This notebook walks you through your first API calls with **Subconscious**, a long-horizon reasoning model.

Subconscious speaks the **OpenAI Chat Completions** protocol, so you use the official `openai` SDK — just pointed at Subconscious's base URL.

To start:

1. Set your `SUBCONSCIOUS_API_KEY` environment variable (or Colab Secret — see cell 1)
2. Click **Run All**

Let's go!

In [ ]:
import os

# Your API key is read from the environment variable SUBCONSCIOUS_API_KEY.
# You can find your key at: https://www.subconscious.dev/platform
#
# In Colab: Secrets pane (key icon) -> add SUBCONSCIOUS_API_KEY
# Locally:  export SUBCONSCIOUS_API_KEY=your_key
api_key = os.environ.get("SUBCONSCIOUS_API_KEY")
if not api_key:
    raise EnvironmentError(
        "SUBCONSCIOUS_API_KEY is not set. "
        "Get your key at https://www.subconscious.dev/platform"
    )

In [ ]:
!pip install openai pydantic > /dev/null 2>&1
# Installs the OpenAI SDK and Pydantic — the only packages this notebook needs.

In [ ]:
# Point the OpenAI SDK at Subconscious. That's the whole setup.
from openai import OpenAI

client = OpenAI(base_url="https://api.subconscious.dev/v1", api_key=api_key)

# There is one model today:
MODEL = "subconscious/tim-qwen3.6-27b"

## *You're in.* Let's try it out!

The simplest thing you can do is send a message and read the reply — exactly like the OpenAI API.

In [ ]:
completion = client.chat.completions.create(
    model=MODEL, extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    messages=[
        {"role": "user", "content": "Explain what an API is in 3 sentences, as if I'm 10 years old."},
    ],
)

print(completion.choices[0].message.content)

## Streaming

Set `stream=True` to print the answer token-by-token as it's generated.

In [ ]:
stream = client.chat.completions.create(
    model=MODEL, extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    messages=[{"role": "user", "content": "Write a short haiku about the ocean."}],
    stream=True,
)

for chunk in stream:
    if chunk.choices:
        print(chunk.choices[0].delta.content or "", end="")

## Structured Output with Pydantic

Sometimes you don't want free text — you want **structured data** you can use in code.

Pass a JSON schema via `response_format` and the reply is guaranteed to match it. We then validate it into a [Pydantic](https://docs.pydantic.dev/) model so the fields are typed.

In [ ]:
from pydantic import BaseModel


class SentimentAnalysis(BaseModel):
    sentiment: str        # "positive", "negative", or "neutral"
    confidence: float     # 0.0 to 1.0
    keywords: list[str]   # words that influenced the sentiment


text_to_analyze = "I absolutely loved the new movie! The acting was superb and it kept me on the edge of my seat."

completion = client.chat.completions.create(
    model=MODEL, extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    messages=[{"role": "user", "content": f"Analyze the sentiment of this text: {text_to_analyze!r}"}],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "SentimentAnalysis",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "sentiment": {"type": "string", "enum": ["positive", "negative", "neutral"]},
                    "confidence": {"type": "number"},
                    "keywords": {"type": "array", "items": {"type": "string"}},
                },
                "required": ["sentiment", "confidence", "keywords"],
                "additionalProperties": False,
            },
        },
    },
)

result = SentimentAnalysis.model_validate_json(completion.choices[0].message.content)
print(f"Sentiment:  {result.sentiment}")
print(f"Confidence: {result.confidence}")
print(f"Keywords:   {result.keywords}")

## Giving the Agent Tools

Want the model to use a tool — search, a database, your own function? Subconscious supports **standard OpenAI function tools**: you pass `tools=[...]`, and when the model wants one it returns `tool_calls`. You run the function and send the result back as a `role:"tool"` message, looping until it has an answer. (Tools run **client-side** — Subconscious doesn't execute them for you.)

Below, the model has no way to know our private employee directory — so it has to call the `lookup_email` tool.

In [ ]:
import json

# A tiny local "database" the model can't possibly know on its own.
EMPLOYEE_DIRECTORY = {"alice": "alice@example.com", "bob": "bob@example.com"}


def lookup_email(name: str) -> str:
    return EMPLOYEE_DIRECTORY.get(name.lower(), "not found")


tools = [{
    "type": "function",
    "function": {
        "name": "lookup_email",
        "description": "Look up an employee's email address by their name.",
        "parameters": {
            "type": "object",
            "properties": {"name": {"type": "string", "description": "The person's name"}},
            "required": ["name"],
        },
    },
}]

messages = [{"role": "user", "content": "What is Alice's email address?"}]

# Reason -> Act -> Observe, until the model answers without calling a tool.
for _ in range(5):
    resp = client.chat.completions.create(model=MODEL, extra_body={"chat_template_kwargs": {"enable_thinking": False}}, messages=messages, tools=tools)
    msg = resp.choices[0].message

    if not msg.tool_calls:
        print("Answer:", msg.content)
        break

    # Record the assistant turn that requested the tool(s).
    # Note: msg.content may be None when the model only returns tool_calls.
    messages.append({
        "role": "assistant",
        "content": msg.content or "",
        "tool_calls": [
            {"id": tc.id, "type": "function",
             "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
            for tc in msg.tool_calls
        ],
    })

    # Run each requested tool and feed the result back as a tool message.
    for tc in msg.tool_calls:
        args = json.loads(tc.function.arguments)
        result = lookup_email(args.get("name", ""))
        print(f"[tool] lookup_email({args}) -> {result}")
        messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

## What's Next?

You've covered the core building blocks: chat, streaming, structured output, and a client-side tool loop. Ideas to explore next:

- **Build a real agent** — the [`hack-cli-starter`](https://github.com/subconscious-systems/subconscious/tree/main/examples/hack-cli-starter) example turns the tool loop above into a full terminal agent with MCP tools.
- **Anthropic-compatible API** — Subconscious also speaks the Anthropic Messages API at `https://api.subconscious.dev`.
- **Scaffold a project** — `npx create-subconscious-app` pulls these examples into a new repo.

### Useful Links

| Resource | Link |
|----------|------|
| Documentation | [docs.subconscious.dev](https://docs.subconscious.dev) |
| API Reference | [docs.subconscious.dev/api-reference](https://docs.subconscious.dev/api-reference/introduction) |
| Platform | [subconscious.dev/platform](https://www.subconscious.dev/platform) |

Happy building!